# Ensemble comparison on Sinhala news: training

Trains and evaluates six models on the LIRNEasia Sinhala misinformation corpus (Jayawickrama et al., 2021, CC BY 4.0): plain bagging, Random Forest, XGBoost, logistic regression, a simple average and stacking. The results are saved as CSV files that my Streamlit dashboard reads.

I ran it on a Colab T4 GPU runtime. The models themselves (scikit-learn and XGBoost with tree_method="hist") run on the CPU.

## 1. Packages
All of these come with Colab.

In [ ]:
import sklearn, xgboost, openpyxl, pandas
print('scikit-learn', sklearn.__version__, '| xgboost', xgboost.__version__, '| pandas', pandas.__version__)

scikit-learn 1.6.1 | xgboost 3.4.1 | pandas 2.2.3


## 2. Google Drive (optional)
With `USE_DRIVE = True` the finished parts are saved to Drive, so the run can continue after a disconnect.

In [ ]:
import os
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/sinhala-misinfo-results'
print('Results folder:', os.environ.get('RESULTS_DIR', 'results/ (in this Colab session)'))

Results folder: results/ (in this Colab session)


## 3. Data
I use Corpus.xlsx because the Sinhala text in Corpus.csv is broken (every letter shows as `?`).

In [ ]:
!mkdir -p data results
!wget -q -O data/Corpus.xlsx https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main/Corpus.xlsx
!ls -la data

total 5620
drwxr-xr-x 2 root root    4096 Nov 22 14:35 .
drwxr-xr-x 1 root root    4096 Nov 22 14:35 ..
-rw-r--r-- 1 root root 5745007 Nov 22 14:35 Corpus.xlsx


## 4. features.py: loading, cleaning, features, models, metrics

Labels: UNCERTAIN = 1, CREDIBLE = 0. PARTIAL (83) and FALSE (27) are dropped because there are too few of them, along with duplicates and texts under 10 words.

Cleaning: outlets naming themselves are masked, old-style spellings standardised, invisible characters removed and links replaced by one token.

Near-copies (cosine similarity ≥ 0.8) get the same group id, so one story is never in both training and test.

Text features are 13 style numbers plus TF-IDF on words and word pairs. I wrote my own token pattern because the default one splits Sinhala words at vowel signs. The source feature is the website, one-hot encoded.

The tree models all use 300 trees, and every model up-weights the smaller class.

In [ ]:
%%writefile features.py
"""Data loading, cleaning, features, models and metrics (imported by train.py)."""
import re

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 42

# Sinhala letters, zero-width joiner, Latin letters and digits.
# \w doesn't work here: it splits Sinhala words at vowel signs.
TOKEN = r"[඀-෿‍A-Za-z0-9]+"
SINHALA = re.compile(r"[඀-෿]")
LATIN = re.compile(r"[A-Za-z]")
DIGIT = re.compile(r"[0-9]")
URL = re.compile(r"https?://|www\.")
WORD = re.compile(TOKEN)
SENT_END = re.compile(r"[.!?]+")

# cleaning rules
# words an outlet uses to name itself (88-100% of matches are in its own articles)
OUTLET_SELF_MENTIONS = (r"බීබීසී|\bBBC\b|News\s*1st|Newsfirst|නිව්ස්\s*ෆස්ට්|\bCOLOMBO\b|සමබිම|ලිපියකි|"
                        r"Vikalpa|\b[iI][mM][aA][gG][eE]\b|Sunanda|Deshapriya")
FULL_URL = r"https?://\S+|www\.\S+"
OLD_SPELLING = "‍්"            # old style, joiner before the virama: "ප‍්‍ර" -> standard "ප්‍ර"
INVISIBLE = r"[​‌﻿]"       # zero-width space, zero-width non-joiner, BOM
MIN_WORDS = 10
NEAR_COPY_SIMILARITY = 0.8

STAT_NAMES = ["chars", "words", "sentences", "avg_word_len", "words_per_sentence",
              "sinhala_share", "latin_share", "digit_share", "exclaims", "questions",
              "quotes", "urls", "unique_word_ratio"]


def clean_text(s):
    """Returns the cleaned texts and how often each rule was applied."""
    counts = {
        "Links replaced by one 'link' token": int(s.str.count(FULL_URL).sum()),
        "Outlet self-mentions masked": 0,
        "Old-style spellings standardised": int(s.str.count(OLD_SPELLING).sum()),
        "Invisible characters removed": int(s.str.count(INVISIBLE).sum()),
    }
    s = s.str.replace(FULL_URL, " https://link ", regex=True)
    counts["Outlet self-mentions masked"] = int(s.str.count(OUTLET_SELF_MENTIONS).sum())
    s = s.str.replace(OUTLET_SELF_MENTIONS, " ", regex=True)
    s = s.str.replace(OLD_SPELLING, "්", regex=False)
    s = s.str.replace(INVISIBLE, "", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s, counts


def load_corpus(path, return_audit=False):
    """Reads and cleans Corpus.xlsx. Label: UNCERTAIN = 1, CREDIBLE = 0."""
    raw = pd.read_excel(path)
    audit = [("Documents in Corpus.xlsx", len(raw), "Starting point (the repo says 3,576; only 3,000 are published)")]
    df = raw[raw["type"].isin(["UNCERTAIN", "CREDIBLE"])].copy()
    audit.append(("Removed: PARTIAL and FALSE labels", len(raw) - len(df), "Too few to evaluate (83 PARTIAL, 27 FALSE)"))
    df["content"] = df["content"].fillna("").astype(str)
    n = len(df)
    df = df[df["content"].str.strip() != ""].drop_duplicates("content")
    audit.append(("Removed: exact duplicate texts", n - len(df), "Same article twice"))

    df["content"], counts = clean_text(df["content"])
    for k, v in counts.items():
        why = {"Links replaced by one 'link' token": "Long URLs became meaningless 'words'",
               "Outlet self-mentions masked": "Text named its own website (e.g. 'COLOMBO (News 1st)'), giving the source away",
               "Old-style spellings standardised": "Same word was counted as two words; the old style is typical of a few websites",
               "Invisible characters removed": "Same word with different hidden characters"}[k]
        audit.append((f"Cleaned: {k.lower()} (occurrences)", v, why))

    n = len(df)
    df = df[df["content"].str.split().str.len() >= MIN_WORDS]
    audit.append((f"Removed: under {MIN_WORDS} words", n - len(df), "Not articles: photo captions, a PDF link, a headline"))

    df["date"] = pd.to_datetime(df["datestamp"])
    df["label"] = (df["type"] == "UNCERTAIN").astype(int)
    df["words"] = df["content"].str.split().str.len()
    df = df[["domain", "date", "type", "label", "words", "content"]].reset_index(drop=True)
    df["group"] = near_copy_groups(df["content"])

    n_grouped = int(len(df) - df["group"].nunique())
    mixed = int((df.groupby("group")["label"].nunique() > 1).sum())
    audit.append(("Kept, grouped: near-copies (similarity ≥ 0.8)", n_grouped,
                  "Same story on several websites, 98% within 3 days; kept in the same fold"))
    audit.append(("Kept: near-copy groups with different labels", mixed, "Real annotation noise; not removed"))
    audit.append(("Final documents", len(df), f"{int(df.label.sum())} UNCERTAIN, {int(len(df) - df.label.sum())} CREDIBLE"))
    audit = pd.DataFrame(audit, columns=["step", "count", "why"])
    return (df, audit) if return_audit else df


def near_copy_groups(texts, threshold=NEAR_COPY_SIMILARITY):
    """Same group id for near-identical texts (labels are not used)."""
    V = TfidfVectorizer(token_pattern=TOKEN).fit_transform(texts)
    S = cosine_similarity(V, dense_output=False) >= threshold
    return connected_components(S, directed=False)[1]


def _stats_one(t):
    words = WORD.findall(t)
    n_chars = max(len(t), 1)
    n_words = max(len(words), 1)
    n_sent = max(len(SENT_END.findall(t)), 1)
    return [len(t), len(words), n_sent,
            sum(map(len, words)) / n_words, len(words) / n_sent,
            len(SINHALA.findall(t)) / n_chars, len(LATIN.findall(t)) / n_chars,
            len(DIGIT.findall(t)) / n_chars, t.count("!"), t.count("?"),
            t.count('"') + t.count("“") + t.count("”") + t.count("'"),
            len(URL.findall(t)), len(set(words)) / n_words]


def text_stats(texts):
    """13 style numbers per article, log-scaled."""
    arr = np.array([_stats_one(t) for t in pd.Series(np.ravel(texts)).astype(str)], dtype=float)
    return np.log1p(arr)


# feature sets for the ablation
FEATURE_SETS = {
    "Text only": ["stats", "tfidf"],
    "Source only": ["source"],
    "Text + Source": ["stats", "tfidf", "source"],
}


def make_features(blocks, max_words=2000):
    parts = {
        "stats": ("stats", make_pipeline(
            FunctionTransformer(text_stats, feature_names_out=lambda *_: np.array(STAT_NAMES)),
            StandardScaler()), "content"),
        "tfidf": ("tfidf", TfidfVectorizer(token_pattern=TOKEN, ngram_range=(1, 2), min_df=3,
                                           max_features=max_words, sublinear_tf=True), "content"),
        "source": ("source", OneHotEncoder(handle_unknown="ignore"), ["domain"]),
    }
    return ColumnTransformer([parts[b] for b in blocks], sparse_threshold=1.0)


def make_model(name, y_train, n_estimators=300, max_depth=None, learning_rate=0.1):
    """Tree models get the same number of trees; all models up-weight the smaller class."""
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                      max_features="sqrt", class_weight="balanced",
                                      n_jobs=-1, random_state=SEED)
    if name == "Bagging (decision trees)":
        # plain bagging: bootstrap rows, all features at every split
        # (RF is the same plus a random feature subset per split)
        return BaggingClassifier(DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
                                 n_estimators=n_estimators, n_jobs=-1, random_state=SEED)
    if name == "XGBoost":
        pos, neg = int(np.sum(y_train)), int(len(y_train) - np.sum(y_train))
        return XGBClassifier(n_estimators=n_estimators, max_depth=max_depth or 6,
                             learning_rate=learning_rate, subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=neg / max(pos, 1), tree_method="hist",
                             eval_metric="logloss", n_jobs=-1, random_state=SEED)
    if name == "Logistic Regression (baseline)":
        return LogisticRegression(max_iter=3000, class_weight="balanced")
    raise ValueError(name)


def make_pipe(model_name, blocks, y_train, **kw):
    return Pipeline([("features", make_features(blocks)),
                     ("model", make_model(model_name, y_train, **kw))])


def metrics(y, p, threshold=0.5):
    y, p = np.asarray(y), np.asarray(p)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y, p)
    top50 = np.argsort(-p)[:50]
    return {
        "ROC-AUC": roc_auc_score(y, p),
        "PR-AUC": average_precision_score(y, p),
        "Accuracy": (tp + tn) / len(y),
        "F1": f1_score(y, pred, zero_division=0),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "False-positive rate": fp / max(fp + tn, 1),
        "TPR @ 10% FPR": float(np.interp(0.10, fpr, tpr)),
        "Precision@50": float(y[top50].mean()),
    }


def top_importances(pipe, k=20):
    names = pipe.named_steps["features"].get_feature_names_out()
    model = pipe.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
    else:  # bagging has no feature_importances_, so average the trees
        imp = np.mean([t.feature_importances_ for t in model.estimators_], axis=0)
    s = pd.Series(imp, index=names).sort_values(ascending=False).head(k)
    s.index = [i.replace("stats__", "[style] ").replace("tfidf__", "[word] ")
               .replace("source__domain_", "[source] ") for i in s.index]
    return s


Writing features.py


## 5. train.py: the experiments

- Main test: stratified 5-fold with near-copies grouped, for 6 models × 3 feature sets (text, source, both).
- Simple average: mean of the RF, XGBoost and LR scores. Stacking: logistic regression on top of RF, XGBoost and LR, trained on inner 3-fold predictions.
- Robustness (text + source): random 5-fold, unseen websites, train up to Sep 2020 and test on Oct–Nov, and a random split of the same size.
- Bootstrap 95% CIs (1,000 resamples of story groups) for four pairs of models.
- Results by article length, stacking weights and feature importance.

TF-IDF and the models are fitted inside each training fold only.

In [ ]:
%%writefile train.py
"""Runs all the experiments and saves the CSVs for the dashboard.

python train.py            skips parts already saved in results/parts/
python train.py --fresh    starts over
Set RESULTS_DIR to save somewhere else (e.g. Google Drive).
"""
import os
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, StratifiedKFold

from features import FEATURE_SETS, SEED, load_corpus, make_pipe, metrics, top_importances

HERE = Path(__file__).parent
OUT = Path(os.environ.get("RESULTS_DIR", HERE / "results"))
PARTS = OUT / "parts"
PARTS.mkdir(parents=True, exist_ok=True)
FRESH = "--fresh" in sys.argv
DATA = HERE / "data" / "Corpus.xlsx"
URL = "https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main/Corpus.xlsx"

BASE = ["Random Forest", "XGBoost", "Logistic Regression (baseline)"]   # stacking / average members
TRAINED = ["Bagging (decision trees)"] + BASE
MODELS = ["Bagging (decision trees)", "Random Forest", "XGBoost", "Logistic Regression (baseline)",
          "Simple average", "Stacking"]
CUTOFF = pd.Timestamp("2020-10-01")
INNER_FOLDS = 3
rng = np.random.default_rng(SEED)

if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    print("Downloading Corpus.xlsx from the LIRNEasia GitHub repo ...")
    urllib.request.urlretrieve(URL, DATA)

df, audit = load_corpus(DATA, return_audit=True)
audit.to_csv(OUT / "audit.csv", index=False)
df[["domain", "date", "type", "label", "words", "group"]].to_csv(OUT / "corpus_overview.csv", index=False)
X, y, groups = df[["content", "domain"]], df["label"].to_numpy(), df["group"].to_numpy()
print(audit.to_string(index=False), "\n")


def fit_fold(tr, te, fs):
    """Trains all models on tr and scores te. Returns scores, times and stacking weights."""
    blocks, scores, secs = FEATURE_SETS[fs], {}, {}
    for m in TRAINED:
        t0 = time.perf_counter()
        pipe = make_pipe(m, blocks, y[tr]).fit(X.iloc[tr], y[tr])
        secs[m] = time.perf_counter() - t0
        scores[m] = pipe.predict_proba(X.iloc[te])[:, 1]

    scores["Simple average"] = np.mean([scores[m] for m in BASE], axis=0)
    secs["Simple average"] = sum(secs[m] for m in BASE)

    # stacking: the meta-model learns from inner out-of-fold scores
    t0 = time.perf_counter()
    inner = np.zeros((len(tr), len(BASE)))
    splitter = StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=SEED)
    for itr, ite in splitter.split(tr, y[tr], groups[tr]):
        a, b = tr[itr], tr[ite]
        for j, m in enumerate(BASE):
            inner[ite, j] = make_pipe(m, blocks, y[a]).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    meta = LogisticRegression(max_iter=1000).fit(inner, y[tr])
    # for the test rows, use the base models trained on all of tr
    scores["Stacking"] = meta.predict_proba(np.column_stack([scores[m] for m in BASE]))[:, 1]
    secs["Stacking"] = time.perf_counter() - t0 + secs["Simple average"]
    weights = dict(zip(BASE, meta.coef_[0]))
    return scores, secs, weights


def run(name, splits, feature_sets, keep_scores=False):
    """Runs one test setup, or loads it if already done."""
    files = {k: PARTS / f"{name}_{k}.csv" for k in ["metrics", "scores", "times", "weights"]}
    if not FRESH and files["metrics"].exists() and (not keep_scores or files["scores"].exists()):
        print(f"  {name}: already done, loading")
        return {k: pd.read_csv(f) for k, f in files.items() if f.exists()}
    rows, score_rows, time_rows, weight_rows = [], [], [], []
    for fs in feature_sets:
        oof = {m: np.full(len(y), np.nan) for m in MODELS}
        for k, (tr, te) in enumerate(splits):
            scores, secs, weights = fit_fold(tr, te, fs)
            for m in MODELS:
                oof[m][te] = scores[m]
                rows.append({"model": m, "features": fs, "fold": k, **metrics(y[te], scores[m])})
                time_rows.append({"model": m, "features": fs, "fold": k, "seconds": secs[m]})
            weight_rows += [{"features": fs, "fold": k, "model": m, "weight": w} for m, w in weights.items()]
            print(f"  {name} · {fs} · fold {k + 1}/{len(splits)} done")
        if keep_scores:
            score_rows += [pd.DataFrame({"model": m, "features": fs, "label": y, "score": oof[m],
                                         "words": df["words"], "group": groups}) for m in MODELS]
    out = {"metrics": pd.DataFrame(rows), "times": pd.DataFrame(time_rows), "weights": pd.DataFrame(weight_rows)}
    if keep_scores:
        out["scores"] = pd.concat(score_rows)
    for k, v in out.items():
        v.to_csv(files[k], index=False)
    return out


# 1. main test: stratified 5-fold, near-copies in the same fold
print("1/5 Main evaluation (near-copies grouped)")
main_splits = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups))
main = run("main", main_splits, FEATURE_SETS, keep_scores=True)
main["metrics"].to_csv(OUT / "cv_metrics.csv", index=False)
main["scores"].to_csv(OUT / "oof_predictions.csv", index=False)
main["times"].to_csv(OUT / "train_times.csv", index=False)
main["weights"].to_csv(OUT / "stacking_weights.csv", index=False)
oof = main["scores"]

# 2. robustness (text + source only)
ROBUST_FS = ["Text + Source"]
print("2/5 Random 5-fold (near-copies may be split)")
r_random = run("random", list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)), ROBUST_FS)
print("3/5 Unseen websites")
r_sites = run("sites", list(GroupKFold(5).split(X, y, df["domain"])), ROBUST_FS)
print("4/5 Temporal test")
old, new = (df["date"] < CUTOFF).to_numpy(), (df["date"] >= CUTOFF).to_numpy()
r_time = run("temporal", [(np.where(old)[0], np.where(new)[0])], ROBUST_FS)
print("5/5 Training-size check (random split, same training size as temporal)")
by_group = pd.Series(np.arange(len(y))).groupby(groups).apply(np.array).to_list()
n_train, size_splits = int(old.sum()), []
for rep in range(3):
    order = rng.permutation(len(by_group))
    take = np.cumsum([len(by_group[g]) for g in order]) <= n_train
    tr = np.sort(np.concatenate([by_group[g] for g in order[take]]))
    size_splits.append((tr, np.setdiff1d(np.arange(len(y)), tr)))
r_size = run("size", size_splits, ROBUST_FS)

robust = pd.concat([
    main["metrics"][main["metrics"].features.isin(ROBUST_FS)].assign(setup="Main: 5-fold, near-copies grouped"),
    r_random["metrics"].assign(setup="Random 5-fold (near-copies may split)"),
    r_sites["metrics"].assign(setup="Unseen websites"),
    r_time["metrics"].assign(setup="Temporal: train ≤ Sep 2020, test Oct–Nov"),
    r_size["metrics"].assign(setup=f"Random split, temporal-size training ({n_train})"),
])
robust.to_csv(OUT / "robustness.csv", index=False)

# 3. bootstrap CIs, resampling whole story groups
mean_auc = main["metrics"].groupby(["features", "model"])["ROC-AUC"].mean()
boot_rows = []
for fs in FEATURE_SETS:
    sc = {m: oof.query("model == @m and features == @fs")["score"].to_numpy() for m in MODELS}
    best = mean_auc[fs][TRAINED].idxmax()
    pairs = [("Bagging vs boosting", "Random Forest", "XGBoost"),
             ("What random features add", "Random Forest", "Bagging (decision trees)"),
             ("Stacking vs best single model", "Stacking", best),
             ("Learned vs simple combination", "Stacking", "Simple average")]
    for label, a, b in pairs:
        for name, fn in [("ROC-AUC", roc_auc_score), ("PR-AUC", average_precision_score)]:
            diffs = []
            for _ in range(1000):
                i = np.concatenate([by_group[g] for g in rng.integers(0, len(by_group), len(by_group))])
                diffs.append(fn(y[i], sc[a][i]) - fn(y[i], sc[b][i]))
            lo, hi = np.percentile(diffs, [2.5, 97.5])
            boot_rows.append({"features": fs, "question": label, "comparison": f"{a} − {b}", "metric": name,
                              "difference": fn(y, sc[a]) - fn(y, sc[b]), "CI low": lo, "CI high": hi,
                              "real difference?": "yes" if lo > 0 or hi < 0 else "no (CI includes 0)"})
pd.DataFrame(boot_rows).to_csv(OUT / "bootstrap_diff.csv", index=False)

# 4. results by article length
bands = pd.qcut(df["words"], 3)
band_names = {b: f"{n} ({int(b.left) + 1}–{int(b.right)} words)"
              for n, b in zip(["Short", "Medium", "Long"], bands.cat.categories)}
len_rows = []
for (m, fs), g in oof.groupby(["model", "features"]):
    for b, idx in pd.Series(np.arange(len(y))).groupby(bands.to_numpy(), observed=True):
        yy, pp = g["label"].to_numpy()[idx.to_numpy()], g["score"].to_numpy()[idx.to_numpy()]
        len_rows.append({"model": m, "features": fs, "length": band_names[b], "documents": len(idx),
                         "UNCERTAIN share": yy.mean(), **metrics(yy, pp)})
pd.DataFrame(len_rows).to_csv(OUT / "by_length.csv", index=False)

# 5. feature importance (one fit on all data)
imp_file = PARTS / "importances.csv"
if FRESH or not imp_file.exists():
    imp_rows = []
    for fs in ["Text + Source", "Text only"]:
        for m in ["Bagging (decision trees)", "Random Forest", "XGBoost"]:
            pipe = make_pipe(m, FEATURE_SETS[fs], y).fit(X, y)
            for feat, val in top_importances(pipe, 20).items():
                imp_rows.append({"model": m, "features": fs, "feature": feat, "importance": val})
    pd.DataFrame(imp_rows).to_csv(imp_file, index=False)
pd.read_csv(imp_file).to_csv(OUT / "importances.csv", index=False)

# summary
pd.set_option("display.width", 220)
print("\nMAIN (mean of 5 folds, near-copies grouped):")
print(main["metrics"].groupby(["features", "model"])[["ROC-AUC", "PR-AUC", "Accuracy", "F1"]].mean().round(3).to_string())
print("\nBootstrap comparisons, ROC-AUC (95% CI):")
print(pd.DataFrame(boot_rows).query("metric == 'ROC-AUC'")[["features", "comparison", "difference", "CI low", "CI high", "real difference?"]].round(3).to_string(index=False))
print("\nStacking weights (mean over folds):")
print(main["weights"].groupby(["features", "model"])["weight"].mean().round(2).unstack().to_string())
print("\nROC-AUC by test setup (Text + Source):")
print(robust.groupby(["setup", "model"])["ROC-AUC"].mean().unstack().round(3).to_string())
print(f"\nSaved results to {OUT}")


Writing train.py


## 6. Run
Prints a line after each fold and skips parts that have already finished.

In [ ]:
!python -u train.py 2>&1 | grep -v -i warning

                                                     step  count                                                                            why
                                 Documents in Corpus.xlsx   3000                 Starting point (the repo says 3,576; only 3,000 are published)
                        Removed: PARTIAL and FALSE labels    110                                     Too few to evaluate (83 PARTIAL, 27 FALSE)
                           Removed: exact duplicate texts     13                                                             Same article twice
Cleaned: links replaced by one 'link' token (occurrences)     66                                           Long URLs became meaningless 'words'
       Cleaned: outlet self-mentions masked (occurrences)    675 Text named its own website (e.g. 'COLOMBO (News 1st)'), giving the source away
  Cleaned: old-style spellings standardised (occurrences)   2953 Same word was counted as two words; the old style is typical of a few w

## 7. Download the results
The CSVs go into the `results` folder of the dashboard.

In [ ]:
import glob, os
from google.colab import files
folder = os.environ.get('RESULTS_DIR', 'results')
csvs = sorted(glob.glob(os.path.join(folder, '*.csv')))
print(len(csvs), 'files:', [os.path.basename(c) for c in csvs])
for c in csvs:
    files.download(c)

10 files: ['audit.csv', 'bootstrap_diff.csv', 'by_length.csv', 'corpus_overview.csv', 'cv_metrics.csv', 'importances.csv', 'oof_predictions.csv', 'robustness.csv', 'stacking_weights.csv', 'train_times.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (11 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>